# Case Study: Does the Lopez-Salido, Stein & Zakrajsek (2017) Sentiment Signal Speak to the 2020-2022 Cycle?
### An out-of-sample probe, made possible by the post-2008 high-yield-share reconstruction

**What this notebook does.** It takes the LSZ (2017) *credit-market sentiment*
first stage — estimated on 1929-2015 — and applies it, unchanged, to the
COVID cycle. The 2020-2021 period looked like the configuration LSZ call
*elevated sentiment*: after the Fed's March-2020 backstop the Baa-Treasury
spread compressed sharply and high-yield issuance surged. Their mechanism says
that froth predicts a subsequent *widening* of spreads (and a slowdown), as
credit conditions mean-revert. The question this notebook asks is simple:
**pointed at 2020-2022, which way does the signal lean, and does it match what
happened?**

**Why this test can now be run at all.** LSZ's sentiment signal has two
ingredients: the lagged *high-yield issuance share* (the froth term) and the
lagged spread *level* (the mean-reversion term). The froth term is the one
that matters most for a story about 2020-21 — and until recently our
high-yield share stopped in 2008 (the hand-transcribed Greenwood-Hanson
series). Issue #3 closed that gap by reconstructing the post-2008 share from
Mergent FISD via WRDS and splicing it on, so the froth variable now reaches
the COVID window. This case study is the payoff of that data work.

| # | Section | What it shows |
|---|---|---|
| 1 | The LSZ first stage on 1929-2015 | The sentiment engine, re-fit; both signs match the paper |
| 2 | Coverage of the froth variable | How far the spliced high-yield share now extends |
| 3 | Out-of-sample probe, 2020-2022 | Full-sentiment vs. spread-only prediction of $\Delta s_t$ |
| 4 | Chart | Predicted vs. realized spread change |
| 5 | What we can and cannot conclude | Splice, sample-size, and generated-regressor caveats |

The verdict is *computed below*, not asserted here: with a three-year window
this is an illustrative probe, not a formal test, and the caveats in Section 5
bound how much weight it can carry.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))
import plot_style as ps
import replicate_table_2 as t2
from settings import config

OUTPUT_DIR = Path(config("OUTPUT_DIR"))

# Annual panel (Baa spread), 1929 through the extension endpoint. Same builder
# the Table II replication uses, so the case study inherits the exact series
# and lag conventions.
df = t2.build_panel(spread_col="BAA_Treasury_spread")

hy = df["ln_hys"].dropna()
print(f"Panel years : {int(df.index.min())}-{int(df.index.max())}")
print(f"ln_hys span : {int(hy.index.min())}-{int(hy.index.max())}  ({hy.notna().sum()} years observed)")

## 1. The LSZ first stage, replicated on 1929-2015

Table II's first stage forecasts the change in the credit spread from
twice-lagged sentiment:

$$\Delta s_t = a_0 + a_1 \ln(\mathrm{HYS})_{t-2} + a_2\, s_{t-2} + u_t.$$

The LSZ signs are $a_1 > 0$ (a high past issuance share predicts the spread
*widening*) and $a_2 < 0$ (a wide spread *mean-reverts*). We fit it on the
replication sample and read the coefficients off, before applying them out of
sample.

In [ ]:
res = t2.run_table_2(df)          # fit on the 1929-2015 replication window
aux = res["aux_spread"]
c = aux.params

print("First-stage  Δs_t = a0 + a1·lnHYS_{t-2} + a2·s_{t-2}   (fit 1929-2015)")
for k in ["const", "ln_hys_lag2", "spread_lag2"]:
    print(f"  {k:>12}: {c[k]:+.4f}   (p = {aux.pvalues[k]:.3f})")

s1 = "high issuance share predicts widening (LSZ sign)" if c["ln_hys_lag2"] > 0 else "UNEXPECTED sign"
s2 = "wide spread mean-reverts (LSZ sign)"              if c["spread_lag2"] < 0 else "UNEXPECTED sign"
print(f"\n  a1 (froth)  {c['ln_hys_lag2']:+.3f}  ->  {s1}")
print(f"  a2 (level)  {c['spread_lag2']:+.3f}  ->  {s2}")

Both signs reproduce the paper's mechanism: the sentiment engine works on the
historical sample. The rest of the notebook holds this 1929-2015 fit *fixed*
and asks what it says about years it never saw.

## 2. The froth variable now reaches past 2008

The hand-transcribed Greenwood-Hanson high-yield share ends in 2008. Issue #3
reconstructed the share from Mergent FISD (via WRDS) for later years and
spliced it on, so the series `t2.build_panel` reads is now continuous through
the recent past. That extension is precisely what lets us form the *full*
two-ingredient sentiment prediction for the COVID window rather than the
spread-reversion leg alone.

The cell below prints how far the froth variable actually reaches in your data
— if the FISD pull has not been run, or coverage is thin, that shows up here
and Section 3 falls back to the spread-only leg automatically.

In [ ]:
last_hys = int(df["ln_hys"].dropna().index.max())
recent = df.loc[2016:2023, ["ln_hys", "spread_lag2", "d_spread", "dy"]].round(3)
print(f"Last year with a high-yield-share observation: {last_hys}")
print(recent)

# Predicting Δs_t needs lnHYS_{t-2}; so the full signal is available for year t
# whenever the froth variable exists at t-2.
covid_years = [y for y in (2020, 2021, 2022) if y in df.index]
full_ok = [yr for yr in covid_years if pd.notna(df.loc[yr, "ln_hys_lag2"])]
print(f"\nCOVID years with a full-sentiment prediction available: {full_ok or 'none'}")

## 3. Out-of-sample probe: 2020-2022

We take the first-stage coefficients fit on 1929-2015 and, without re-fitting,
form the predicted spread change for each COVID year:

$$\widehat{\Delta s_t} = a_0 + a_1 \ln(\mathrm{HYS})_{t-2} + a_2\, s_{t-2}.$$

We report two versions side by side: the **full sentiment** prediction (both
ingredients) and the **spread-reversion leg only** ($a_0 + a_2 s_{t-2}$, the
fragment the earlier data-limited version of this study was stuck with). The
test is directional: does the sign of the predicted change match the sign of
the realized change?

In [ ]:
def predict_dspread(year, coeffs, use_froth):
    # Predicted change in the spread from the fixed 1929-2015 first stage.
    # use_froth=True adds the lnHYS_{t-2} term; if that value is missing, the
    # full-sentiment prediction is undefined (NaN) for the year.
    pred = coeffs["const"] + coeffs["spread_lag2"] * df.loc[year, "spread_lag2"]
    if use_froth:
        h = df.loc[year, "ln_hys_lag2"]
        if pd.isna(h):
            return np.nan
        pred = pred + coeffs["ln_hys_lag2"] * h
    return pred


rows = []
for yr in covid_years:
    rows.append({
        "year": yr,
        "s_(t-2)": round(df.loc[yr, "spread_lag2"], 2),
        "lnHYS_(t-2)": (round(df.loc[yr, "ln_hys_lag2"], 3)
                        if pd.notna(df.loc[yr, "ln_hys_lag2"]) else np.nan),
        "pred_full": round(predict_dspread(yr, c, True), 3)
                     if pd.notna(predict_dspread(yr, c, True)) else np.nan,
        "pred_spread_only": round(predict_dspread(yr, c, False), 3),
        "actual_dS": round(df.loc[yr, "d_spread"], 3),
    })
scorecard = pd.DataFrame(rows).set_index("year")
print(scorecard)


def hit_rate(col):
    ok = n = 0
    for yr in covid_years:
        p, a = scorecard.loc[yr, col], scorecard.loc[yr, "actual_dS"]
        if pd.notna(p) and pd.notna(a):
            n += 1
            ok += int(np.sign(p) == np.sign(a))
    return ok, n


print()
for col, label in [("pred_full", "full sentiment (froth + reversion)"),
                   ("pred_spread_only", "spread-reversion leg only")]:
    ok, n = hit_rate(col)
    verdict = f"correct direction in {ok}/{n} years" if n else \
              "no usable years (froth variable missing in window)"
    print(f"  {label:>34}: {verdict}")

How to read the scorecard: a row "hits" when the predicted and realized
changes in the spread share a sign. The contrast between the two prediction
columns is the point of the whole exercise — the spread-only leg mechanically
predicts "widening toward the mean" in almost every year the spread sits below
average, so on its own it can be right for the wrong reason. The froth term is
what gives the signal genuine content about *sentiment*, and it is only
available because of the post-2008 reconstruction.

A caveat that belongs right next to the number: **three years cannot carry
statistical weight.** This is an illustrative direction check, not the formal
out-of-sample test the full sample would support.

In [ ]:
ps.set_paper_style()
fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = np.arange(len(covid_years))
w = 0.27
ax.bar(x - w, scorecard["pred_full"], w,
       label=r"Predicted $\Delta s$ (full sentiment)", color="#1f4e79")
ax.bar(x, scorecard["pred_spread_only"], w,
       label=r"Predicted $\Delta s$ (spread leg only)", color="#8faadc")
ax.bar(x + w, scorecard["actual_dS"], w,
       label=r"Actual $\Delta s$", color=ps.HIGHLIGHT_COLOR)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in covid_years])
ax.set_ylabel("Change in Baa-Treasury spread (pp)")
ax.set_title("LSZ sentiment out of sample: predicted vs. realized " r"$\Delta s$, 2020-2022")
ps.style_axes(ax)
ax.legend()
fig.tight_layout()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "case_study_covid_oos.pdf")
plt.show()

## 4. What we can and cannot conclude

Four honest caveats bound the reading above:

1. **The splice is not seamless.** The froth values in the COVID window come
   from the FISD reconstruction, whose construction (Moody's-rated
   denominator, coverage starting in the early 1980s) differs from the
   pre-2009 transcribed series the first stage was largely estimated on. The
   ingredient we feed the model out of sample is measured a little differently
   from the one it learned on — a genuine limitation, not a bug.
2. **Three years is illustrative, not inferential.** A direction check on
   2020-2022 cannot distinguish skill from luck; it can only show which way the
   signal leaned.
3. **Generated-regressor uncertainty.** The prediction plugs in first-stage
   fitted coefficients; the standard two-step "plug-in" procedure used here
   does not propagate their sampling error, so any apparent precision is
   overstated (the same caveat noted in `replicate_table_2`).
4. **The first stage is held fixed at 1929-2015.** This is a true out-of-sample
   application; re-fitting through the window would change the coefficients and
   is a different (in-sample) exercise.

The contribution, then, is not "the framework called COVID." It is that the
test can now be *run at all* — the froth ingredient that the earlier version of
this study had to leave out is in place — and that, run honestly, the signal
should be read with the caveats above rather than as vindication.

## Summary

We re-fit the LSZ sentiment first stage on 1929-2015 (both signs match the
paper), confirm that the post-2008 FISD/WRDS reconstruction now extends the
high-yield share into the COVID window, and use the fixed historical fit to
form an out-of-sample prediction of the 2020-2022 spread changes. The scorecard
in Section 3 reports the realized hit rate for the full-sentiment signal and
for the spread-only leg; the chart in Section 4 shows them against reality.
With only three years and a spliced froth series, the result is an honest
directional probe rather than a formal test — but it is a probe that, thanks to
issue #3, no longer has to omit the variable that matters most.